# Tilli Lead Scoring Pipeline

This notebook loads school lead data, normalizes it for consistency, and prepares it for downstream lead scoring using the Anthropic API. It serves as the entry point for cleaning raw school records before enrichment and scoring.

In [2]:
import json
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Project root: Notebooks/ -> parent, or current dir if already at root
BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "Notebooks":
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "Data"
OUTPUT_DIR = BASE_DIR / "Output"
ENV_PATH = BASE_DIR / ".env"
DATA_PATH = DATA_DIR / "mock_schools.csv"
SCORED_CSV_PATH = OUTPUT_DIR / "scored_schools.csv"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

load_dotenv(ENV_PATH)

from research import normalize_columns, research_school


In [3]:
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
print(df.head())


Shape: (50, 7)
                             school_name location  size board_affiliation  \
0         St. Mary's Convent High School   Mumbai  1850              ICSE   
1                 Bombay Scottish School   Mumbai  2100              ICSE   
2  Dhirubhai Ambani International School   Mumbai  1200                IB   
3             Podar International School   Mumbai  1450             IGCSE   
4                      Rajhans Vidyalaya   Mumbai  1650              CBSE   

  city_tier school_type  founded_year  
0    Tier 1     Private          1962  
1    Tier 1     Private          1958  
2    Tier 1     Private          2003  
3    Tier 1     Private          2010  
4    Tier 1     Private          1975  


In [4]:
df = normalize_columns(df)
df["size"] = pd.to_numeric(df["size"], errors="coerce")
df = df.fillna("Unknown")
print(df)


                                          school_name    location  size  \
0                      St. Mary's Convent High School      Mumbai  1850   
1                              Bombay Scottish School      Mumbai  2100   
2               Dhirubhai Ambani International School      Mumbai  1200   
3                          Podar International School      Mumbai  1450   
4                                   Rajhans Vidyalaya      Mumbai  1650   
5                        Kendriya Vidyalaya IIT Powai      Mumbai  2200   
6                   The Mother's International School       Delhi  1800   
7                      Delhi Public School R.K. Puram       Delhi  2500   
8                                  Springdales School       Delhi  1900   
9                              Sardar Patel Vidyalaya       Delhi   850   
10                                 The British School       Delhi   720   
11                               St. Columba's School       Delhi  1450   
12                 Nation

## Research and scoring


In [5]:
FORCE_REFRESH = True

test_row = df.iloc[0]
print(f"Testing on: {test_row['school_name']}")

result = research_school(test_row, force_refresh=FORCE_REFRESH)
print(json.dumps(result, indent=2))


Testing on: St. Mary's Convent High School
{
  "score": 45,
  "conversion_likelihood": "Low",
  "reasoning": "Research found the actual St. Mary's Convent High School Mulund, Mumbai, which is affiliated with Maharashtra State Board (not ICSE as stated in the brief). The school has a strong foundation in holistic development and shows alignment with some SEL principles through its stated mission. However, specific evidence of formal SEL programmes, dedicated wellbeing initiatives, or technology adoption is limited. The school was ranked among top girls' schools by EducationWorld (2021-22), indicating good reputation. There is mention of computerised science labs and a well-resourced library, suggesting some tech readiness. However, no public evidence of comprehensive edtech platforms, principal's public statements on SEL/wellbeing, or specific counselling programmes was found. The school participated in a School Leaders Partnership programme in 2017, showing some openness to collaborati

In [6]:
results = []
web_search_count = 0
cache_count = 0
FORCE_REFRESH = True  # Set True to force fresh web search for every school

for _, row in df.iterrows():
    result = research_school(row, force_refresh=FORCE_REFRESH)

    if result.get("_from_cache"):
        cache_count += 1
    elif result.get("_web_search_used"):
        web_search_count += 1

    record = row.to_dict()
    record.update({
        "score": result["score"],
        "conversion_likelihood": result["conversion_likelihood"],
        "reasoning": result.get("reasoning", ""),
        "evidence": result.get("evidence", []),
        "opportunities": result.get("opportunities", ""),
        "concerns": result.get("concerns", ""),
        "confidence": result.get("confidence", ""),
    })
    results.append(record)
    print(f"{row['school_name']}: {result['score']} ({result['conversion_likelihood']})")

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("score", ascending=False).reset_index(drop=True)
results_df["rank"] = range(1, len(results_df) + 1)

OUTPUT_DIR.mkdir(exist_ok=True)
results_df.to_csv(SCORED_CSV_PATH, index=False)

print(f"\nSaved {len(results_df)} schools to {SCORED_CSV_PATH}")
print(f"Web search used for {web_search_count} schools; loaded from cache for {cache_count} schools.")
results_df.head(10)

St. Mary's Convent High School: 55 (Medium)
Bombay Scottish School: 68 (Medium)
Dhirubhai Ambani International School: 72 (Medium)
Podar International School: 68 (Medium)
Rajhans Vidyalaya: 62 (Medium)
Kendriya Vidyalaya IIT Powai: 62 (Medium)
The Mother's International School: 68 (Medium)
Delhi Public School R.K. Puram: 62 (Medium)
Springdales School: 71 (Medium)
Sardar Patel Vidyalaya: 61 (Medium)
The British School: 72 (Medium)
St. Columba's School: 48 (Low)
National Public School Indiranagar: 35 (Low)
Inventure Academy: 78 (High)
Vidya Niketan Public School: 38 (Low)
Canadian International School: 68 (Medium)
Kendriya Vidyalaya ASC Centre: 48 (Low)
Bishop Cotton Girls' School: 48 (Low)
La Martiniere Girls' Higher Secondary School: 0 (Very Low)
Vidya Mandir Senior Secondary School: 56 (Medium)
PSBB Senior Secondary School: 54 (Medium)
The Hindu Senior Secondary School: 48 (Low)
American International School Chennai: 78 (High)
Chettinad Vidyashram: 68 (Medium)
Oakridge International 

,school_name,location,size,board_affiliation,city_tier,school_type,founded_year,score,conversion_likelihood,reasoning,evidence,opportunities,concerns,confidence,rank
0,American International School Chennai,Chennai,480,IB,Tier 1,Private,1997,78,High,AISC demonstrates strong alignment with Tilli'...,[School website features dedicated Wellbeing F...,AISC's existing commitment to wellbeing and st...,AISC already operates a well-developed in-hous...,high,1
1,Chirec International School,Hyderabad,1580,IGCSE,Tier 1,Private,1989,78,High,Chirec International School demonstrates stron...,[School website explicitly states it brings to...,Exceptional fit for outreach. The school has d...,"School already has a documented SEL programme,...",high,2
2,Inventure Academy,Bangalore,1100,ICSE,Tier 1,Private,2005,78,High,Inventure Academy demonstrates exceptionally s...,[School has a dedicated Student Safety & Well-...,Exceptional fit. Direct outreach to Nooraine F...,"School already operates a comprehensive, well-...",high,3
3,Oakridge International School,Hyderabad,1050,IB,Tier 1,Private,2001,73,Medium,Oakridge International School demonstrates str...,[School website states 'social and emotional l...,Strong foundation of SEL awareness and wellbei...,"School already has comprehensive, embedded SEL...",mid,4
4,Strawberry Fields High School,Chandigarh,680,IGCSE,Tier 2,Private,2006,72,Medium,Strawberry Fields High School demonstrates str...,[School website explicitly states education ex...,Strong foundation of existing emotional-securi...,"School already has comprehensive, philosophy-d...",mid,5
5,Dhirubhai Ambani International School,Mumbai,1200,IB,Tier 1,Private,2003,72,Medium,Dhirubhai Ambani International School demonstr...,[School has a dedicated Student Care Services ...,Direct alignment possible through emphasizing ...,School already operates comprehensive counsell...,mid,6
6,The British School,Delhi,720,IGCSE,Tier 1,Private,1963,72,Medium,The British School Delhi demonstrates signific...,"[School has documented counselling suite, medi...",Position Tilli as enhancing rather than replac...,"School already has visible, documented SEL pro...",mid,7
7,Springdales School,Delhi,1900,CBSE,Tier 1,Private,1955,71,Medium,Springdales School demonstrates substantial al...,[School website explicitly states commitment t...,Strong strategic fit: Springdales' explicit co...,No evidence of current external SEL platform p...,mid,8
8,Delhi Public School Chandigarh,Chandigarh,2000,CBSE,Tier 2,Private,2004,68,Medium,Delhi Public School Chandigarh demonstrates si...,[School website and Education World profile ex...,Strong cultural fit given school's explicit fo...,School already has established counselling and...,mid,9
9,St. John's High School,Chandigarh,1150,ICSE,Tier 2,Private,1959,68,Medium,"St. John's High School is a well-established, ...",[School website documents 'counselling and lea...,Strong institutional values alignment: The sch...,No documented formal SEL programme found—the s...,mid,10
